# 🗺️ Civic Data Analysis - Interactive Mapping Tool

## Explore Electoral Data for Vancouver & Toronto

This Jupyter notebook provides an interactive visualization of election data across multiple electoral boundaries (Federal, Provincial, Municipal) and years (2008-2021).

### Features:
- **Interactive Maps**: Hover over divisions to see voter statistics
- **Multiple Views**: Standard map, satellite imagery, and heat maps
- **Dynamic Data**: Switch between years and boundary types
- **Historical Analysis**: Compare electoral patterns across multiple election cycles

### How to Use:
1. Run all cells (Kernel → Restart & Run All)
2. Use the dropdown widgets to select boundary type
3. Choose year and visualization mode
4. Interact with the map (zoom, pan, hover for info)

---

## Cell 1: Import Required Libraries

In [ ]:
import geopandas as gpd
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from ipyleaflet import Map, GeoData, WidgetControl, LocalTileLayer
from ipywidgets import HTML, Layout
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Cell 2: Configuration & File Paths

Define all file paths in one central location for easy management.

In [ ]:
# ============================================================================
# FILE CONFIGURATION - UPDATE THESE PATHS IF YOUR DATA IS LOCATED ELSEWHERE
# ============================================================================

# Shapefile paths (GeoJSON or Shapefiles in ZIP format)
SHAPEFILE_PATHS = {
    'municipal': {
        2008: "ShapeFiles/Municipal/municipal2008.zip",
        2011: "ShapeFiles/Municipal/municipal2011.zip",
    },
    'federal': {
        2015: "ShapeFiles/Federal/fed2015.zip",
        2019: "ShapeFiles/Federal/fed2019.zip",
        2021: "ShapeFiles/Federal/fed2021.zip",
    },
    'provincial': {
        2013: "ShapeFiles/Provincial/pro2013.zip",
        2017: "ShapeFiles/Provincial/pro2017.zip",
    }
}

# Data file paths (Excel format)
DATA_PATHS = {
    'municipal': {
        2008: "DataFiles/Municipal/Municipal-2008.xlsx",
        2011: "DataFiles/Municipal/Municipal-2011.xlsx",
    },
    'federal': {
        2015: "DataFiles/Federal/Federal_Data_2015.xlsx",
        2019: "DataFiles/Federal/Federal_Data_2019.xlsx",
        2021: "DataFiles/Federal/Federal_Data_2021.xlsx",
    },
    'provincial': {
        2013: None,  # 2013 data not available
        2017: "DataFiles/Provincial/Provincial_2017.xlsx",
    }
}

# Map configuration
MAP_CONFIG = {
    'center': (49.26742200000007, -123.10661199999998),  # Vancouver, BC
    'zoom': 12,
    'scroll_wheel_zoom': True,
    'layout': Layout(width='100%', height='600px')
}

# Style configuration for map features
FEATURE_STYLE = {
    'color': 'black',
    'fillColor': '#366370',
    'opacity': 0.05,
    'weight': 1.9,
    'dashArray': '2',
    'fillOpacity': 0.6,
}

HOVER_STYLE = {
    'fillColor': '#b08a3e',
    'fillOpacity': 0.9
}

# Satellite tile source
SATELLITE_TILE = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"

print("✅ Configuration loaded successfully!")

## Cell 3: Utility Functions

Core functions for loading data, creating maps, and generating visualizations.

In [ ]:
def load_geospatial_data(boundary_type: str, year: int) -> dict:
    """
    Load and merge shapefile with electoral data.
    
    Args:
        boundary_type: 'municipal', 'federal', or 'provincial'
        year: Election year
    
    Returns:
        Dictionary containing merged GeoDataFrame and metadata
    """
    try:
        shapefile_path = SHAPEFILE_PATHS[boundary_type][year]
        data_path = DATA_PATHS[boundary_type][year]
        
        if data_path is None:
            return {
                'status': 'error',
                'message': f'Data not available for {boundary_type} {year}',
                'data': None
            }
        
        # Load shapefile and data
        print(f"📂 Loading shapefile: {shapefile_path}")
        shapefile = gpd.read_file(shapefile_path)
        
        print(f"📊 Loading data: {data_path}")
        data = pd.read_excel(data_path)
        
        # Handle type conversion for provincial data
        if boundary_type == 'provincial' and 'VA_CODE' in shapefile.columns:
            shapefile['VA_CODE'] = shapefile['VA_CODE'].astype(int)
        
        # Merge data with shapefile
        print("🔗 Merging data with shapefile...")
        merged = pd.merge(data, shapefile)
        
        print(f"✅ Successfully loaded {len(merged)} features")
        
        return {
            'status': 'success',
            'data': merged,
            'boundary_type': boundary_type,
            'year': year
        }
    
    except Exception as e:
        print(f"❌ Error loading data: {str(e)}")
        return {
            'status': 'error',
            'message': str(e),
            'data': None
        }


def create_interactive_map(merged_data: pd.DataFrame, boundary_type: str, year: int, satellite: bool = False) -> Map:
    """
    Create interactive leaflet map with electoral data.
    
    Args:
        merged_data: Merged GeoDataFrame
        boundary_type: Type of boundary
        year: Election year
        satellite: Show satellite imagery
    
    Returns:
        ipyleaflet Map object
    """
    # Create map
    map_obj = Map(
        center=MAP_CONFIG['center'],
        zoom=MAP_CONFIG['zoom'],
        scroll_wheel_zoom=MAP_CONFIG['scroll_wheel_zoom']
    )
    
    # Add satellite layer if requested
    if satellite:
        map_obj.add_layer(LocalTileLayer(path=SATELLITE_TILE))
    
    # Convert to GeoDataFrame
    geo_data = gpd.GeoDataFrame(merged_data, geometry='geometry')
    
    # Create GeoData layer
    geo_layer = GeoData(
        geo_dataframe=geo_data,
        style=FEATURE_STYLE,
        hover_style=HOVER_STYLE,
        geometry='geometry'
    )
    
    # Add layer to map
    map_obj.add(geo_layer)
    
    # Create info widget
    info_widget = HTML(f"<b>{boundary_type.title()} - {year}</b><br>Hover over a region to see details")
    info_widget.layout.margin = "0px 20px 20px 20px"
    
    # Define hover callback based on boundary type
    def create_hover_callback(info_widget, boundary_type):
        def update_info(feature, **kwargs):
            props = feature['properties']
            
            if boundary_type == 'municipal':
                info_widget.value = f"""<b>Municipal Division</b><br>
                Division: {props.get('division', 'N/A')}<br>
                Name: {props.get('name', 'N/A')}<br>
                Percentage: {props.get('percentage', 'N/A')}%"""
            
            elif boundary_type == 'federal':
                pd_num = props.get('PD_NUM') or props.get('PDNUM', 'N/A')
                votes = props.get('total votes') or props.get('Total votes', 'N/A')
                info_widget.value = f"""<b>Federal Division</b><br>
                Division: {pd_num}<br>
                Total Votes: {votes}<br>
                Percentage: {props.get('percentage', 'N/A')}%"""
            
            elif boundary_type == 'provincial':
                info_widget.value = f"""<b>Provincial Division</b><br>
                Division: {props.get('VA_CODE', 'N/A')}<br>
                Registered Votes: {props.get('registered_votes', 'N/A')}<br>
                Percentage: {props.get('percentage', 'N/A')}%"""
        
        return update_info
    
    # Attach hover callback
    geo_layer.on_hover(create_hover_callback(info_widget, boundary_type))
    
    # Add control widget
    control = WidgetControl(widget=info_widget, position="topright")
    map_obj.add(control)
    
    return map_obj


def create_heatmap(merged_data: pd.DataFrame, boundary_type: str, year: int) -> None:
    """
    Create choropleth heatmap visualization.
    
    Args:
        merged_data: Merged GeoDataFrame
        boundary_type: Type of boundary
        year: Election year
    """
    try:
        geo_data = gpd.GeoDataFrame(merged_data, geometry='geometry')
        
        # Determine location column based on boundary type
        if boundary_type == 'municipal':
            location_col = 'division'
            color_col = 'color schema'
        elif boundary_type == 'federal':
            location_col = 'PD_NUM' if 'PD_NUM' in merged_data.columns else 'PDNUM'
            color_col = 'Color Schema'
        else:  # provincial
            location_col = 'VA_CODE'
            color_col = 'color schema'
        
        # Create choropleth
        fig = px.choropleth(
            merged_data,
            locations=location_col,
            color=color_col,
            color_continuous_scale="Viridis",
            scope="north america",
            title=f"{boundary_type.title()} Electoral Engagement - {year}"
        )
        
        fig.update_geos(fitbounds="locations", visible=False)
        fig.show()
    
    except Exception as e:
        print(f"❌ Error creating heatmap: {str(e)}")

print("✅ All utility functions defined successfully!")

## Cell 4: Main Interactive Widget

This is the main interactive interface. Use the dropdown menus to explore the data!

In [ ]:
def election_explorer(boundary_type: str) -> None:
    """
    Main interactive explorer function.
    
    Args:
        boundary_type: 'Federal', 'Municipal', or 'Provincial'
    """
    
    # Normalize input
    boundary_type = boundary_type.lower()
    
    # Get available years for this boundary type
    if boundary_type in SHAPEFILE_PATHS:
        available_years = sorted(SHAPEFILE_PATHS[boundary_type].keys())
    else:
        print(f"❌ Invalid boundary type: {boundary_type}")
        return
    
    # Create year selector
    year_selector = widgets.Dropdown(
        options=available_years,
        description='Election Year:',
        style={'description_width': '120px'}
    )
    
    # Create visualization mode selector
    mode_selector = widgets.RadioButtons(
        options=['Standard Map', 'Satellite View', 'Heat Map'],
        description='View Mode:',
        style={'description_width': '120px'}
    )
    
    # Create output widget
    output = widgets.Output()
    
    def on_selection_change(change):
        with output:
            output.clear_output(wait=True)
            
            year = year_selector.value
            mode = mode_selector.value
            
            print(f"\n📍 {boundary_type.upper()} Elections - {year}")
            print(f"📺 Mode: {mode}")
            print("\n" + "="*50)
            
            # Load data
            result = load_geospatial_data(boundary_type, year)
            
            if result['status'] != 'success':
                print(f"⚠️  {result['message']}")
                return
            
            merged_data = result['data']
            
            # Create visualization
            if mode == 'Standard Map':
                print("\n🗺️  Creating standard map...")
                map_obj = create_interactive_map(merged_data, boundary_type, year, satellite=False)
                display(map_obj)
            
            elif mode == 'Satellite View':
                print("\n🛰️  Creating satellite view...")
                map_obj = create_interactive_map(merged_data, boundary_type, year, satellite=True)
                display(map_obj)
            
            elif mode == 'Heat Map':
                print("\n🔥 Creating heat map...")
                create_heatmap(merged_data, boundary_type, year)
    
    # Attach event listeners
    year_selector.observe(on_selection_change, names='value')
    mode_selector.observe(on_selection_change, names='value')
    
    # Create controls box
    controls = widgets.VBox([year_selector, mode_selector])
    
    # Display interface
    print(f"\n🎯 {boundary_type.upper()} Election Data Explorer")
    print("="*50)
    display(controls)
    display(output)
    
    # Trigger initial display
    on_selection_change(None)


# Create main selector
print("\n" + "="*60)
print("🗺️  CIVIC DATA ANALYSIS - INTERACTIVE ELECTION EXPLORER")
print("="*60)
print("\nSelect a boundary type to begin exploring electoral data.\n")

boundary_selector = widgets.ToggleButtons(
    options=['Federal', 'Municipal', 'Provincial'],
    description='Boundary Type:',
    style={'description_width': '120px'},
    button_style='info'
)

output_main = widgets.Output()

def on_boundary_change(change):
    with output_main:
        output_main.clear_output(wait=True)
        election_explorer(change.new)

boundary_selector.observe(on_boundary_change, names='value')

display(boundary_selector)
display(output_main)

# Trigger initial display
on_boundary_change(type('obj', (object,), {'new': 'Federal'})())

---

## 📊 About This Project

This interactive mapping tool visualizes civic engagement patterns across Canada using election data from:

- **Federal Elections**: 2015, 2019, 2021
- **Provincial Elections**: 2017
- **Municipal Elections**: 2008, 2011

### Data Metrics Displayed:
- Total registered voters
- Voter participation rates
- Electoral division boundaries
- Geographic analysis

### Technologies Used:
- **GeoPandas**: Geospatial data manipulation
- **Plotly**: Interactive visualizations
- **ipyleaflet**: Interactive mapping
- **Jupyter**: Interactive notebooks

### Created by: Kuldeep Singh (@iamyamraj)
Last Updated: April 2026